# Limits & Continuity
**Topic:** Calculus for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** what a limit is in plain English: what a function is approaching, not what it equals
- **Identify** whether a function is continuous at a point by checking three conditions
- **Interpret** why discontinuous loss functions cannot be minimized with gradient descent

> **Tip:** Start by moving the **x-value slider** toward the limit point from the left side and watch the output approach a value, then approach from the right. If both sides agree, the limit exists.

---
## How we got here

In *01: Functions and Graphs* we learned that a function maps inputs to outputs and that the shape of the graph carries information. Now we ask a more precise question: what happens to the output of a function as the input gets very, very close to a particular value? This question, and the answer to it, is the foundation on which all of calculus is built.

---
## Why this matters for data science

Gradient descent works by taking small steps along a loss function. For that to work, the loss function must be continuous: no sudden jumps, no holes, no gaps. When you use the cross-entropy loss for classification or MSE for regression, you are choosing functions that are continuous everywhere, which is a deliberate design decision. Understanding continuity helps you understand why some loss functions are differentiable and others are not, and why that distinction matters when choosing between models.

---
## Try it yourself

In [ ]:
x_fine = np.linspace(-2.0, 4.0, 1000)

out = Output()

FN_OPTIONS = [
    "Continuous — limit exists and equals f(1)",
    "Removable discontinuity — limit exists, f(1) undefined",
    "Jump discontinuity — left ≠ right, no limit",
    "Vertical asymptote — function diverges at x=1",
]

fn_dd = Dropdown(options=FN_OPTIONS, value=FN_OPTIONS[0],
    description="Function:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="520px"))

x_slider = FloatSlider(value=0.0, min=-1.9, max=3.9, step=0.05,
    description="x position:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"))

FN_CONFIG = {
    FN_OPTIONS[0]: {
        "fn":     lambda x: x ** 2,
        "label":  "f(x) = x²",
        "L_left": 1.0, "L_right": 1.0, "f_at_a": 1.0,
        "y_range": [-1, 8],
    },
    FN_OPTIONS[1]: {
        "fn":     lambda x: np.where(np.abs(x - 1) < 1e-9, np.nan, (x**2 - 1) / (x - 1)),
        "label":  "f(x) = (x²−1)/(x−1)  [hole at x=1]",
        "L_left": 2.0, "L_right": 2.0, "f_at_a": None,
        "y_range": [-1, 6],
    },
    FN_OPTIONS[2]: {
        "fn":     lambda x: np.where(x < 1, x, x + 2.0),
        "label":  "f(x) = x for x<1,  x+2 for x≥1",
        "L_left": 1.0, "L_right": 3.0, "f_at_a": 3.0,
        "y_range": [-3, 7],
    },
    FN_OPTIONS[3]: {
        "fn":     lambda x: np.where(np.abs(x - 1) < 0.04, np.nan, 1 / (x - 1)),
        "label":  "f(x) = 1/(x−1)",
        "L_left": None, "L_right": None, "f_at_a": None,
        "y_range": [-10, 10],
    },
}

def render(fn_name, x_val):
    cfg    = FN_CONFIG[fn_name]
    fn     = cfg["fn"]
    a      = 1.0
    y_rng  = cfg["y_range"]

    y_curve = np.clip(fn(x_fine), y_rng[0], y_rng[1])

    try:
        fx = float(fn(np.array([x_val]))[0])
    except Exception:
        fx = np.nan
    fx_clipped = np.clip(fx, y_rng[0], y_rng[1]) if not np.isnan(fx) else np.nan

    from_left  = x_val < a
    dot_color  = PALETTE["primary"] if from_left else PALETTE["secondary"]
    side_str   = "left" if from_left else "right"

    Ll, Lr = cfg["L_left"], cfg["L_right"]
    if Ll is not None and Lr is not None and abs(Ll - Lr) < 1e-9:
        limit_str = f"Limit as x→{a} = {Ll:.1f}"
    elif Ll is not None and Lr is not None:
        limit_str = f"Left limit = {Ll:.1f},  Right limit = {Lr:.1f}  →  no limit"
    else:
        limit_str = f"Function diverges as x→{a}  →  no limit"

    fx_str = f"{fx:.4f}" if not np.isnan(fx) else "undefined"

    traces = [
        go.Scatter(x=x_fine, y=y_curve, mode="lines",
            line=dict(color=PALETTE["muted"], width=2.5), name=cfg["label"]),
        go.Scatter(x=[x_val], y=[fx_clipped] if not np.isnan(fx_clipped) else [],
            mode="markers",
            marker=dict(color=dot_color, size=14, symbol="circle"),
            name=f"x = {x_val:.2f}  ({side_str} approach)  f(x) = {fx_str}"),
        go.Scatter(x=[a], y=[Ll] if Ll is not None else [],
            mode="markers",
            marker=dict(color=PALETTE["accent"], size=12,
                        symbol="circle" if cfg["f_at_a"] is not None else "circle-open",
                        line=dict(width=2, color=PALETTE["accent"])),
            name=f"x = {a}  (limit point)"),
    ]

    title = (f"x = {x_val:.2f}, approaching {a} from the {side_str}  |  "
             f"f(x) = {fx_str}  |  {limit_str}")
    layout = base_layout(title=title, xaxis_title="x", yaxis_title="f(x)")
    layout.update(yaxis=dict(range=y_rng), xaxis=dict(range=[-2, 4]))
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces, layout=layout)))

def on_change(change): render(fn_dd.value, x_slider.value)
fn_dd.observe(on_change, names="value")
x_slider.observe(on_change, names="value")
display(VBox([fn_dd, x_slider, out]))
render(fn_dd.value, x_slider.value)

---
## What's happening?

A limit asks: as the input x gets closer and closer to some value (call it a), what value does the output f(x) get closer and closer to? We write this as:

The **limit of f(x) as x approaches a** equals L.

| Concept | Plain English | Formal notation |
|---------|-------------|----------------|
| Limit from the left | What f approaches as x sneaks up from below | lim(x→a⁻) f(x) |
| Limit from the right | What f approaches as x sneaks up from above | lim(x→a⁺) f(x) |
| The limit exists | Both sides agree on the same value | lim(x→a) f(x) = L |
| Continuity at a | Three things: f(a) defined, limit exists, they are equal | f(a) = lim(x→a) f(x) |

$$\lim_{x \to a} f(x) = L$$

This reads: "as x gets arbitrarily close to a, f(x) gets arbitrarily close to L." Notice it says nothing about what f(a) actually equals. The limit is about the journey, not the destination.

### Three conditions for continuity

A function is continuous at the point x = a if all three hold:
1. f(a) is defined (no hole, no asymptote at that point)
2. The limit as x approaches a exists (left and right sides agree)
3. The limit equals the function value: lim(x→a) f(x) = f(a)

If any one of the three fails, the function has a discontinuity there. Return to the widget and check each function type against these three conditions.

---
## Real-world example: Why MSE is preferred over 0-1 loss for training

The 0-1 loss counts the fraction of predictions that are exactly wrong. It sounds like a natural thing to minimize. The problem is it is not continuous: it jumps from 0 to 1 when a prediction crosses the decision boundary, and it is flat everywhere else. The chart shows why this makes gradient descent impossible on 0-1 loss versus why MSE and cross-entropy work.

Notice:

- **Notice:** The 0-1 loss is flat almost everywhere, with vertical jumps at decision boundaries; the derivative is zero everywhere except at those jumps, so gradient descent has no direction to move in
- **Notice:** MSE is a smooth parabola; at every point there is a well-defined slope that tells gradient descent which way is downhill
- **Notice:** This is why training accuracy (a 0-1 loss) and training loss (cross-entropy or MSE) are tracked separately; you optimize the continuous surrogate loss, and you measure the discontinuous metric

> **Discussion question:** Decision trees split on thresholds that optimize Gini impurity or entropy, not a differentiable loss. Does this mean decision trees do not need continuity? What optimization technique do they use instead of gradient descent?

In [3]:
x = np.linspace(-3, 3, 500)

mse_loss = x ** 2
zero_one = (x < 0).astype(float)
ce_loss  = np.log(1 + np.exp(-2 * x))

fig = make_subplots(rows=1, cols=3,
    subplot_titles=["0-1 Loss — untrainable", "MSE — trainable", "Cross-Entropy — trainable"],
    horizontal_spacing=0.1)

for col, (y, color, name) in enumerate([
    (zero_one, PALETTE["secondary"], "0-1 loss"),
    (mse_loss, PALETTE["primary"],   "MSE loss"),
    (ce_loss,  PALETTE["accent"],    "Cross-entropy"),
], start=1):
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines",
                             line=dict(color=color, width=2.5), name=name),
                  row=1, col=col)

fig.update_layout(
    title=dict(text="Continuity Determines Trainability — Same Error, Three Loss Functions",
               font=dict(size=FONT["size_title"])),
    paper_bgcolor=PALETTE["background"], plot_bgcolor=PALETTE["surface"],
    height=400, showlegend=False,
)
fig.update_xaxes(title_text="Prediction margin", gridcolor="#DEE2E6")
fig.update_yaxes(title_text="Loss value", gridcolor="#DEE2E6")
fig.show()

### Limits and continuity in ML context

| Concept | Plain English | Why it matters for ML |
|---------|-------------|----------------------|
| Limit exists | Both sides approach the same value | Required for a derivative to exist at that point |
| Continuous function | No jumps, holes, or asymptotes | Required for gradient descent to have a direction at every point |
| Differentiable | Has a well-defined slope at every point | Stronger than continuity; needed for gradient-based optimization |
| Lipschitz continuous | Rate of change is bounded everywhere | Guarantees gradient descent converges with a small enough learning rate |

---
## Key takeaway

> **A limit describes what a function approaches, not what it equals; continuity means no sudden jumps, and it is the minimum requirement for gradient descent to function at all.**

---
*Next up: Derivatives — the tool that turns "what is this function approaching?" into "how fast is it changing right here?"*